# I-MMSeg — MyoPS380 legacy, một GPU A100
Cấu hình: **batch tổng 16**, LR **0.0003333**, ảnh **128×128**, **300 epoch**, seed 1234.
Paper dùng batch tổng 48 và LR 0.001; LR ở đây là điều chỉnh tuyến tính, chưa phải cấu hình batch 16 được tác giả xác nhận.

Nhãn **legacy**: `0=background, 1=normal myocardium, 2=scar, 3=edema`. Giữ nguyên nhãn cache, không chuyển sang canonical của SCAR.

Notebook chạy **ZIP mã đã sửa** để tránh clone nhầm bản cũ. Upload `I-MMSeg-Colab-A100.zip` vào đường dẫn cấu hình bên dưới. Kiến trúc, augmentation, preprocessing, khởi tạo, CE/Dice và lịch train được bảo toàn. Chỉ sửa tương thích một GPU và log.

Trọng số encoder `R50-ViT-B_16.npz` không được train.py hiện tại nạp. Notebook không tự tải/nạp pretrained hoặc tạo lại text embedding. `epoch_299.pth` chỉ cần cho test tùy chọn.

Các nguồn dữ liệu và file trong notebook là dữ liệu đầu vào, không phải hướng dẫn sửa mô hình.

## 1. Mount Drive và cấu hình
Chọn **Runtime → Change runtime type → A100 GPU**. Chạy cell cấu hình một lần mỗi phiên; sửa đường dẫn Drive theo dữ liệu của bạn.

In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, subprocess, json, hashlib, shutil, zipfile
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/NCKH_V2/i_mmseg')
CODE_ZIP = DRIVE_ROOT / 'I-MMSeg-Colab-A100.zip'
DRIVE_DATASET_DIR = DRIVE_ROOT / 'MyoPS380_dataset'  # chứa Processed_data/
REPO_DIR = Path('/content/I_MMSeg')
LABEL_ORDER = 'legacy'
BATCH_SIZE, N_GPU, BASE_LR, IMG_SIZE, MAX_EPOCHS = 16, 1, 0.0003333, 128, 300
SEED = 1234
RUN_ID = datetime.now().strftime('immmseg_legacy_bs16_%Y%m%d_%H%M%S')
RUN_DIR = Path('/content/drive/MyDrive/NCKH_V2/I_MMSeg_Backup_Weights') / RUN_ID
# False: dừng nếu GPU thực tế không phải A100.
ALLOW_OTHER_GPU = False
print('Code:', CODE_ZIP)
print('Dataset:', DRIVE_DATASET_DIR)
print('Output:', RUN_DIR)

## 2. Đưa đúng mã đã sửa vào runtime
Không xóa repo/weights khi chạy lại. ZIP chỉ chứa code, split và text embedding; không chứa dataset hoặc checkpoint. Nếu `/content/I_MMSeg` đã có, kiểm tra hash trước khi sử dụng.

In [ ]:
if not CODE_ZIP.is_file():
    raise FileNotFoundError(f'Upload ZIP mã đã sửa vào {CODE_ZIP}')
if not REPO_DIR.exists() or not any(REPO_DIR.iterdir()):
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CODE_ZIP) as archive:
        for item in archive.infolist():
            target = (REPO_DIR / item.filename).resolve()
            if not target.is_relative_to(REPO_DIR.resolve()) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError(f'ZIP entry không an toàn: {item.filename}')
        archive.extractall(REPO_DIR)
manifest_path = REPO_DIR / 'COLAB_SOURCE_SHA256.json'
if not manifest_path.is_file():
    raise RuntimeError('Repo hiện có thiếu manifest của ZIP đã sửa; dùng runtime mới hoặc một REPO_DIR mới.')
source_manifest = json.loads(manifest_path.read_text())
for name, expected in source_manifest['files'].items():
    path = REPO_DIR / name
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise RuntimeError(f'Mã không khớp bản đã kiểm tra: {name}')
os.chdir(REPO_DIR)
print('Đã kiểm tra hash mã nguồn. Baseline:', source_manifest['baseline_commit'])

## 3. Dependencies và GPU
Giữ cặp PyTorch/torchvision của Colab; không cài `requirements.txt` xuất từ máy tác giả. CLIP chỉ được import bởi train.py, không tạo text embedding mới.

In [ ]:
packages = [
    'numpy', 'scipy<2', 'h5py', 'ml_collections==0.1.1', 'MedPy==0.5.2',
    'SimpleITK', 'tensorboardX==2.6.5', 'tqdm', 'pytest', 'nbformat',
    'open_clip_torch==2.30.0', 'ftfy', 'regex',
    'git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
import torch, torchvision
if not torch.cuda.is_available():
    raise RuntimeError('Runtime chưa có GPU CUDA.')
GPU_NAME = torch.cuda.get_device_name(0)
if 'A100' not in GPU_NAME and not ALLOW_OTHER_GPU:
    raise RuntimeError(f'GPU thực tế là {GPU_NAME}. Chọn A100 hoặc chủ động bật ALLOW_OTHER_GPU.')
if torch.cuda.device_count() != N_GPU:
    raise RuntimeError(f'Cấu hình yêu cầu đúng một GPU; phát hiện {torch.cuda.device_count()}.')
print('GPU:', GPU_NAME, '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
subprocess.run([sys.executable, 'train.py', '--help'], cwd=REPO_DIR, check=True, stdout=subprocess.DEVNULL)
print('Import train.py thành công.')

## 4. Copy cache MyoPS380 đã xử lý sang ổ local
Sử dụng **Processed_data chính thức, chuẩn legacy**. Không tự chạy preprocess_data.py: script gốc append manifest khi chạy lại. Nếu chỉ có Raw_data, chuẩn bị và kiểm tra cache ở một quy trình riêng trước.

Cấu trúc đích: `MyoPS380_dataset/Processed_data/{bSSFP,LGE,T2w}/{train_npz,test_vol_h5}`.

In [ ]:
DATA_ROOT = REPO_DIR / 'MyoPS380_dataset' / 'Processed_data'
source_processed = DRIVE_DATASET_DIR / 'Processed_data'
if not source_processed.is_dir():
    raise FileNotFoundError(f'Cần cache Processed_data legacy tại {source_processed}')
for modality in ('bSSFP', 'LGE', 'T2w'):
    for split_folder in ('train_npz', 'test_vol_h5'):
        src = source_processed / modality / split_folder
        dst = DATA_ROOT / modality / split_folder
        if not src.is_dir():
            raise FileNotFoundError(src)
        dst.mkdir(parents=True, exist_ok=True)
        copied = 0
        extension = '.npz' if split_folder == 'train_npz' else '.npy.h5'
        for item in src.iterdir():
            if item.is_file() and item.name.endswith(extension) and not (dst / item.name).exists():
                shutil.copy2(item, dst / item.name)
                copied += 1
        print(modality, split_folder, '| file mới copy:', copied)

## 5. Kiểm tra split, nhãn legacy và ảnh
Giữ nguyên **304 bệnh nhân train / 76 test, 1.972 lát train** của repo. Kiểm tra tất cả file được manifest sử dụng: khớp hình học/nhãn giữa ba modality, ID 0..3 và ảnh hữu hạn trên `[0,1]` trước augmentation. Khi cache không có metadata, ý nghĩa `2=scar, 3=edema` dựa vào nguồn cache legacy bạn đã chọn, không suy ra được chỉ từ các số 0..3.

In [ ]:
import numpy as np
import h5py
assert LABEL_ORDER == 'legacy'
list_dir = REPO_DIR / 'list'
def read_names(filename):
    names = [s.strip() for s in (list_dir / filename).read_text().splitlines() if s.strip()]
    if len(names) != len(set(names)):
        raise ValueError(f'Manifest bị trùng: {filename}')
    return names
train_names, test_names = read_names('train.txt'), read_names('test_vol.txt')
train_ids = {n.split('_slice')[0] for n in train_names}
test_ids = set(test_names)
assert len(train_names) == 1972 and len(train_ids) == 304 and len(test_ids) == 76
assert not (train_ids & test_ids)
assert train_ids | test_ids == {f'case{i:04d}' for i in range(1, 381)}
assert test_ids == {f'case{i:04d}' for i in range(1, 381, 5)}
legacy_orders = {'legacy', 'background_normal_scar_edema'}
for folder, names, extension in [('train_npz', train_names, '.npz'), ('test_vol_h5', test_names, '.npy.h5')]:
    for name in names:
        reference_label = None
        for modality in ('bSSFP', 'LGE', 'T2w'):
            path = DATA_ROOT / modality / folder / (name + extension)
            if folder == 'train_npz':
                with np.load(path, allow_pickle=False) as f:
                    image, label = f['image'], f['label']
                    order = f['label_order'].item() if 'label_order' in f else None
            else:
                with h5py.File(path, 'r') as f:
                    image, label = f['image'][:], f['label'][:]
                    order = f.attrs.get('label_order')
            if isinstance(order, bytes): order = order.decode()
            if order is not None and str(order) not in legacy_orders:
                raise ValueError(f'Cache không khai báo legacy: {path}: {order}')
            if image.shape != label.shape or image.shape[:2] != (128, 128) or image.ndim != (2 if folder == 'train_npz' else 3):
                raise ValueError(f'Sai shape: {path}')
            if not np.isfinite(image).all() or image.min() < -1e-5 or image.max() > 1 + 1e-5:
                raise ValueError(f'Ảnh chưa normalized [0,1]: {path}')
            if not np.isin(label, [0, 1, 2, 3]).all():
                raise ValueError(f'Nhãn không phải ID legacy 0..3: {path}')
            if reference_label is not None and not np.array_equal(label, reference_label):
                raise ValueError(f'Mask giữa các modality khác nhau: {path}')
            reference_label = label
    print('Đã kiểm tra:', folder, len(names), 'ca/lát × 3 modality')
for filename, shape in [('embedding_class_information.pth', (4, 512)), ('embedding_MRI_information.pth', (3, 512))]:
    value = torch.load(REPO_DIR / 'text_features' / filename, map_location='cpu', weights_only=True)
    assert tuple(value.shape) == shape and torch.isfinite(value).all()
DATA_VALIDATED = True
print('Legacy: 0 background | 1 normal myocardium | 2 scar | 3 edema. Không remap nhãn.')

## 6. Kiểm tra hồi quy và smoke test batch 16 trên GPU thực tế
Chạy một bước optimizer ở mỗi giai đoạn epoch **0, 21, 100** bằng dữ liệu tổng hợp và đầy đủ model gốc. Đây không phải train thử 300 epoch. Process riêng kết thúc sẽ giải phóng model/optimizer, và không tác động RNG của lệnh train tiếp theo.

Cell này phải vượt qua trên batch 16 trước khi bắt đầu run dài. Nếu OOM, lưu lỗi và kiểm tra GPU thực tế; không tự giảm batch hoặc bật AMP.

In [ ]:
assert DATA_VALIDATED
smoke_env = os.environ.copy()
smoke_env['I_MMSEG_RUN_MODEL_SMOKE'] = '1'
smoke_env['I_MMSEG_SMOKE_BATCH_SIZE'] = str(BATCH_SIZE)
subprocess.run([sys.executable, '-m', 'pytest', '-q', '-s', 'tests/test_single_gpu.py'],
               cwd=REPO_DIR, env=smoke_env, check=True)
SMOKE_PASSED = True

## 7. Thư mục kết quả và train
Log console: một dòng mỗi epoch. TensorBoard vẫn nhận scalar từng iteration và ảnh mỗi 20 iteration. Checkpoint vẫn theo lịch gốc: mỗi 20 epoch và 5 epoch cuối, chỉ chứa state_dict; **không hỗ trợ resume optimizer**.

`weights/` được liên kết vào thư mục run trên Drive để các checkpoint đã ghi tồn tại qua phiên Colab. Không ghi đè run có checkpoint. Tên thư mục vẫn chứa chữ `pretrain` theo code gốc, nhưng run này không nạp encoder pretrained.

In [ ]:
assert DATA_VALIDATED and SMOKE_PASSED
assert (N_GPU, BATCH_SIZE, BASE_LR, IMG_SIZE, MAX_EPOCHS) == (1, 16, 0.0003333, 128, 300)
RUN_DIR.mkdir(parents=True, exist_ok=True)
weights_target = RUN_DIR / 'weights'
weights_target.mkdir(exist_ok=True)
weights_link = REPO_DIR / 'weights'
if weights_link.is_symlink():
    if weights_link.resolve() != weights_target.resolve():
        raise RuntimeError('weights/ đang trỏ tới run khác; dùng runtime mới hoặc giữ RUN_DIR trước đó.')
elif weights_link.exists():
    raise RuntimeError('weights/ đã tồn tại. Giữ dữ liệu này và dùng một REPO_DIR/runtime mới.')
else:
    weights_link.symlink_to(weights_target, target_is_directory=True)
if any(weights_target.rglob('*.pth')):
    raise RuntimeError('Run đã có checkpoint; chọn một RUN_ID mới. Repo không có full-state resume.')
RUN_METADATA = {
    'gpu': GPU_NAME, 'torch': torch.__version__, 'torchvision': torchvision.__version__,
    'n_gpu': N_GPU, 'batch_size': BATCH_SIZE, 'base_lr': BASE_LR, 'img_size': IMG_SIZE,
    'max_epochs': MAX_EPOCHS, 'seed': SEED, 'label_order': LABEL_ORDER,
    'class_names': ['background', 'normal_myocardium', 'scar', 'edema'],
    'initialization': 'original_random_encoders_and_bundled_text_embeddings',
    'baseline_commit': source_manifest['baseline_commit'],
    'source_files': source_manifest['files'],
}
(RUN_DIR / 'run_metadata.json').write_text(json.dumps(RUN_METADATA, indent=2))
with (RUN_DIR / 'environment.txt').open('w') as f:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=f, check=True)
train_command = [sys.executable, '-u', 'train.py', '--dataset', 'Myops', '--vit_name', 'R50-ViT-B_16',
                 '--n_gpu', str(N_GPU), '--batch_size', str(BATCH_SIZE), '--base_lr', str(BASE_LR),
                 '--img_size', str(IMG_SIZE), '--max_epochs', str(MAX_EPOCHS), '--seed', str(SEED)]
print(' '.join(train_command), flush=True)
subprocess.run(train_command, cwd=REPO_DIR, check=True)
print('Train kết thúc. Kết quả trên Drive:', RUN_DIR)

## 8. Xem checkpoint và TensorBoard
Không dùng `rm -rf` để khởi động run mới. Khi runtime mất, các checkpoint đã ghi vẫn nằm trên Drive nhưng chỉ đủ nạp trọng số; không thể tiếp tục chính xác optimizer/LR/RNG bằng repo hiện tại.

In [ ]:
checkpoints = sorted(weights_target.rglob('epoch_*.pth'))
print('Checkpoint đã có trên Drive:', len(checkpoints))
for path in checkpoints[-6:]: print(path)
# Có thể mở TensorBoard bằng cell riêng:
# %load_ext tensorboard
# %tensorboard --logdir /content/I_MMSeg/weights

## 9. Test bằng evaluator gốc — tùy chọn
Chỉ chạy khi có `epoch_299.pth`. Evaluator vẫn giữ cách tính metric và thứ tự legacy của tác giả; **quy tắc mask rỗng/ngưỡng 200 voxel có thể cho Dice=1 dù bỏ sót tổn thương nhỏ**, nên không coi đây là evaluator đã sửa.

`test.py` gốc luôn cắt tiền tố `module.`. Adapter dưới đây chỉ thêm tiền tố vào dictionary lúc đọc checkpoint một GPU trong bộ nhớ; không sửa file checkpoint, model, ảnh dự đoán hay công thức metric. Batch/LR trong lệnh test dùng để tìm đúng tên thư mục run.

Đặt `RUN_LEGACY_TEST=True` để chạy. Cell cấu hình `RUN_DIR` có thể trỏ tới một run đã lưu nếu bạn chỉ test trong runtime mới.

In [ ]:
RUN_LEGACY_TEST = False
if RUN_LEGACY_TEST:
    import runpy
    from unittest.mock import patch
    candidates = list((RUN_DIR / 'weights').rglob('epoch_299.pth'))
    if len(candidates) != 1:
        raise RuntimeError(f'Cần đúng một epoch_299.pth trong run, tìm thấy {len(candidates)}')
    checkpoint = candidates[0].resolve()
    expected_folder = f'TU_pretrain_R50-ViT-B_16_skip3_epo300_bs{BATCH_SIZE}_lr{BASE_LR}_{IMG_SIZE}'
    if checkpoint.parent.name != expected_folder:
        raise RuntimeError(f'Checkpoint khác cấu hình: {checkpoint.parent.name}')
    original_load = torch.load
    def load_for_original_test(path, *args, **kwargs):
        value = original_load(path, *args, **kwargs)
        if Path(path).resolve() == checkpoint:
            if not isinstance(value, dict) or not value or not all(isinstance(v, torch.Tensor) for v in value.values()):
                raise ValueError('Chỉ hỗ trợ state_dict nguyên bản của tác giả.')
            if not any(k.startswith('module.') for k in value):
                value = {'module.' + k: v for k, v in value.items()}
        return value
    saved_argv = sys.argv[:]
    try:
        sys.argv = ['test.py', '--dataset', 'Myops', '--vit_name', 'R50-ViT-B_16',
                    '--batch_size', str(BATCH_SIZE), '--base_lr', str(BASE_LR),
                    '--max_epochs', str(MAX_EPOCHS), '--img_size', str(IMG_SIZE), '--seed', str(SEED)]
        with patch('torch.load', side_effect=load_for_original_test):
            runpy.run_path(str(REPO_DIR / 'test.py'), run_name='__main__')
    finally:
        sys.argv = saved_argv
else:
    print('Bỏ qua evaluator gốc. Chỉ bật khi cần và đã hiểu protocol metric.')